
# Pharmacy Sales Analysis with Python

**Author:** Hussieni Gamal  
**Period:** 2014-01-02 to 2019-10-08  
**Purpose:** Explore pharmaceutical sales trends across daily, weekly, monthly, and hourly datasets.

This notebook demonstrates:
- Data loading and validation
- Data cleaning and feature engineering
- Exploratory data analysis
- Time-based aggregation
- Category performance analysis
- Correlation analysis
- Business insight generation


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 5.5)

DATA_DIR = Path("../data")
IMAGE_DIR = Path("../images")
DRUG_COLS = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]


## 1. Load the datasets

In [ ]:

daily = pd.read_csv(DATA_DIR / "salesdaily.csv")
weekly = pd.read_csv(DATA_DIR / "salesweekly.csv")
monthly = pd.read_csv(DATA_DIR / "salesmonthly.csv")
hourly = pd.read_csv(DATA_DIR / "saleshourly.csv")

print("Daily:", daily.shape)
print("Weekly:", weekly.shape)
print("Monthly:", monthly.shape)
print("Hourly:", hourly.shape)
daily.head()


## 2. Data quality checks

In [ ]:

quality_summary = pd.DataFrame({
    "dtype": daily.dtypes.astype(str),
    "missing_values": daily.isna().sum(),
    "unique_values": daily.nunique()
})
quality_summary


## 3. Cleaning and feature engineering

In [ ]:

daily["date"] = pd.to_datetime(daily["datum"], errors="coerce")
daily = daily.sort_values("date").reset_index(drop=True)

daily["year"] = daily["date"].dt.year
daily["month"] = daily["date"].dt.month
daily["month_name"] = daily["date"].dt.month_name().str[:3]
daily["quarter"] = daily["date"].dt.to_period("Q").astype(str)
daily["weekday"] = daily["date"].dt.day_name()
daily["total_sales"] = daily[DRUG_COLS].sum(axis=1)

q33, q66 = daily["total_sales"].quantile([0.33, 0.66])
daily["performance_band"] = pd.cut(
    daily["total_sales"],
    bins=[-np.inf, q33, q66, np.inf],
    labels=["Low", "Medium", "High"]
)

daily.head()


## 4. Descriptive statistics

In [ ]:

descriptive_stats = daily[DRUG_COLS + ["total_sales"]].describe().T
descriptive_stats["coefficient_of_variation"] = (
    descriptive_stats["std"] / descriptive_stats["mean"]
)
descriptive_stats.sort_values("mean", ascending=False)


## 5. Category contribution

In [ ]:

category_totals = daily[DRUG_COLS].sum().sort_values(ascending=False)
category_summary = pd.DataFrame({
    "total_sales": category_totals,
    "share_pct": category_totals / category_totals.sum() * 100
})
category_summary.round(2)


In [ ]:

category_totals.sort_values().plot(kind="barh")
plt.title("Total Pharmaceutical Sales by Drug Category")
plt.xlabel("Total units sold")
plt.ylabel("Drug category")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## 6. Annual trend

In [ ]:

yearly_sales = daily.groupby("year")["total_sales"].sum()
yearly_sales


In [ ]:

plt.plot(yearly_sales.index, yearly_sales.values, marker="o", linewidth=2)
plt.title("Annual Pharmaceutical Sales Trend")
plt.xlabel("Year")
plt.ylabel("Total units sold")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 7. Monthly seasonality

In [ ]:

monthly_avg = daily.groupby("month")["total_sales"].mean()
monthly_labels = [pd.Timestamp(2000, m, 1).strftime("%b") for m in monthly_avg.index]

plt.bar(monthly_labels, monthly_avg.values)
plt.title("Average Daily Sales by Month")
plt.xlabel("Month")
plt.ylabel("Average daily units")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 8. Day-of-week analysis

In [ ]:

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_avg = daily.groupby("weekday")["total_sales"].mean().reindex(weekday_order)

plt.bar([day[:3] for day in weekday_order], weekday_avg.values)
plt.title("Average Daily Sales by Day of Week")
plt.xlabel("Day of week")
plt.ylabel("Average daily units")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 9. Hourly demand pattern

In [ ]:

hourly["date"] = pd.to_datetime(hourly["datum"], errors="coerce")
hourly["hour"] = hourly["date"].dt.hour
hourly["total_sales"] = hourly[DRUG_COLS].sum(axis=1)

hourly_avg = hourly.groupby("hour")["total_sales"].mean()
plt.plot(hourly_avg.index, hourly_avg.values, marker="o")
plt.title("Average Sales by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Average units sold")
plt.xticks(hourly_avg.index)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 10. Category seasonality

In [ ]:

monthly["date"] = pd.to_datetime(monthly["datum"], errors="coerce")
monthly["month"] = monthly["date"].dt.month
monthly_by_drug = monthly.groupby("month")[DRUG_COLS].mean()
monthly_by_drug.index = [pd.Timestamp(2000, m, 1).strftime("%b") for m in monthly_by_drug.index]

for col in DRUG_COLS:
    plt.plot(monthly_by_drug.index, monthly_by_drug[col], marker="o", label=col)

plt.title("Average Monthly Sales Pattern by Drug Category")
plt.xlabel("Month")
plt.ylabel("Average monthly units")
plt.legend(ncol=2, frameon=False)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 11. Correlation analysis

In [ ]:

correlation_matrix = daily[DRUG_COLS].corr()
correlation_matrix.round(2)


In [ ]:

plt.figure(figsize=(8.5, 7))
im = plt.imshow(correlation_matrix.values, aspect="auto")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(DRUG_COLS)), DRUG_COLS, rotation=45, ha="right")
plt.yticks(range(len(DRUG_COLS)), DRUG_COLS)
plt.title("Correlation Between Drug Categories")

for i in range(len(DRUG_COLS)):
    for j in range(len(DRUG_COLS)):
        plt.text(j, i, f"{correlation_matrix.iloc[i, j]:.2f}",
                 ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()


## 12. Daily performance classification

In [ ]:

band_summary = daily.groupby("performance_band", observed=False)["total_sales"].agg(
    days="count",
    average_sales="mean",
    minimum_sales="min",
    maximum_sales="max"
)
band_summary.round(2)



## 13. Key business insights

1. **N02BE** is the dominant category and contributes approximately **49.4%** of total recorded sales.
2. The strongest annual result occurred in **2016**, with approximately **25,235 units** sold.
3. **January** records the highest average daily sales, while **July** records the lowest.
4. **Saturday** is the strongest weekday based on average daily demand.
5. Hourly data indicates that demand peaks around **19:00**.
6. Drug categories show different seasonal patterns, supporting category-specific stocking and replenishment decisions.
7. The performance-band model provides a simple operational method to flag unusually strong or weak sales days.


## 14. Export cleaned analytical dataset

In [ ]:

output_columns = [
    "date", *DRUG_COLS, "year", "month", "month_name",
    "quarter", "weekday", "total_sales", "performance_band"
]
daily[output_columns].to_csv(DATA_DIR / "salesdaily_cleaned.csv", index=False)
print("Cleaned dataset exported successfully.")
